# Benchmark GraphRAG vs RAG Vectoriel — G3 EPITA 2026

Comparaison sur un subset HotpotQA (500 questions multi-hop).
Métriques : F1 token-level, Exact Match.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
from dotenv import load_dotenv
load_dotenv("../.env")

import numpy as np
import matplotlib.pyplot as plt
from graphrag_core.llm import get_llm_client
from graphrag_core.extractor import extract_triples
from graphrag_core.graph import build_knowledge_graph, KnowledgeGraph
from graphrag_core.pipeline import run as graphrag_run

In [ ]:
DATA = Path("../data/hotpotqa")
qa_pairs = json.loads((DATA / "qa_pairs.json").read_text())
sample_files = sorted(DATA.glob("sample_*.txt"))
docs_map = {f.name: f.read_text(encoding="utf-8") for f in sample_files[:50]}
print(f"{len(qa_pairs)} questions, {len(docs_map)} documents chargés")

In [ ]:
llm = get_llm_client()
all_triples = []
for fname, text in docs_map.items():
    all_triples.extend(extract_triples(text, llm))
    print(f"  {fname}: {len(all_triples)} triplets total")
kg = build_knowledge_graph(all_triples, docs_map)
print(f"\nKG: {kg.nx_graph.number_of_nodes()} entités, {kg.nx_graph.number_of_edges()} relations, {len(kg.communities)} communautés")

In [ ]:
def f1_score(pred: str, gold: str) -> float:
    """Compute token-level F1 score between prediction and gold answer.

    Args:
        pred: The predicted answer string.
        gold: The gold (ground truth) answer string.

    Returns:
        F1 score in [0.0, 1.0]. Returns 0.0 if either string is empty
        or if there are no common tokens.
    """
    pred_tokens = set(pred.lower().split())
    gold_tokens = set(gold.lower().split())
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = pred_tokens & gold_tokens
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens)
    r = len(common) / len(gold_tokens)
    return 2 * p * r / (p + r)

def exact_match(pred: str, gold: str) -> float:
    """Check if prediction exactly matches the gold answer (case-insensitive, stripped).

    Args:
        pred: The predicted answer string.
        gold: The gold (ground truth) answer string.

    Returns:
        1.0 if the strings match exactly after stripping and lowercasing, 0.0 otherwise.
    """
    return float(pred.strip().lower() == gold.strip().lower())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = list(docs_map.values())
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(corpus)

def rag_vectoriel(question: str, top_k: int = 3) -> str:
    """Retrieve top-k documents via TF-IDF and generate an answer using the LLM.

    Args:
        question: The natural language question to answer.
        top_k: Number of top documents to retrieve by cosine similarity.

    Returns:
        The LLM's answer string based on the retrieved context.
    """
    q_vec = vectorizer.transform([question])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    context = "\n\n".join(corpus[i][:500] for i in top_indices)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer concisely:"
    return llm.complete(prompt)

In [ ]:
N = 50
questions = qa_pairs[:N]

results = {"graphrag": {"f1": [], "em": []}, "rag_v": {"f1": [], "em": []}}

for i, qa in enumerate(questions):
    q, gold = qa["question"], qa["answer"]
    # GraphRAG
    gr = graphrag_run(q, kg, llm, docs_map)
    results["graphrag"]["f1"].append(f1_score(gr.answer, gold))
    results["graphrag"]["em"].append(exact_match(gr.answer, gold))
    # RAG vectoriel
    rv = rag_vectoriel(q)
    results["rag_v"]["f1"].append(f1_score(rv, gold))
    results["rag_v"]["em"].append(exact_match(rv, gold))
    if i % 10 == 0:
        print(f"  {i}/{N} questions traitées")

print("\n=== Résultats ===")
for method, scores in results.items():
    print(f"{method}: F1={np.mean(scores['f1']):.3f}, EM={np.mean(scores['em']):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
methods = ["GraphRAG", "RAG Vectoriel"]
f1_means = [np.mean(results["graphrag"]["f1"]), np.mean(results["rag_v"]["f1"])]
em_means = [np.mean(results["graphrag"]["em"]), np.mean(results["rag_v"]["em"])]
colors = ["#6366f1", "#94a3b8"]

axes[0].bar(methods, f1_means, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_title("F1 Score (token-level)", fontweight="bold")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("F1")

axes[1].bar(methods, em_means, color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_title("Exact Match", fontweight="bold")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("EM")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor("#f8faff")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("GraphRAG vs RAG Vectoriel — HotpotQA (50 questions)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../docs/benchmark_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
hop_counts = [len(r.trace) for r in [graphrag_run(qa["question"], kg, llm, docs_map) for qa in questions[:20]]]
plt.figure(figsize=(6, 3))
plt.hist(hop_counts, bins=range(0, max(hop_counts)+2), color="#6366f1", alpha=0.85, edgecolor="white")
plt.title("Distribution du nombre de hops par réponse GraphRAG")
plt.xlabel("Nombre de hops")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()